In [1]:
spark.sql("CREATE SCHEMA IF NOT EXISTS Retailorg_gold")


StatementMeta(, e210f3f9-ba26-43bf-b859-3d0953d6d8b1, 3, Finished, Available, Finished, False)

DataFrame[]

# Dokumentation: Gold-Faktentabelle (fact_sales_order_items)

**Layer:** Gold Layer (Business / Reporting Schicht)  
**Ziel:** Konsolidierung aller Verkaufsaktivitäten, Marketing-Rabatte und Web-Tracking-Daten in einer zentralen, hochperformanten Faktentabelle. Diese Tabelle bildet das Herzstück des Sternschemas (Star Schema) für das Umsatz- und Marketing-Reporting.

---

## 1. Daten-Zusammenführung (E2E Integration)
Die Tabelle führt vier Datenströme aus dem Silver Layer zusammen, um einen vollständigen 360-Grad-Blick auf jede Verkaufstransaktion zu ermöglichen:
*   **`sales_orders` & `sales_order_items` (Basis):** Zusammenführung von Auftragskopf und Auftragspositionen (Header-Item-Join).
*   **`sales_promotion` (Marketing):** Left-Join über `sales_order_id` und `product_id`, um gewährte Kampagnenrabatte direkt an der Artikelzeile anzureichern.
*   **`sales_clicked_items` (Web-Analytics):** Left-Join, um das Klickverhalten (`click_value`) aus dem Onlineshop mit dem tatsächlichen Kauf abzugleichen.

---

## 2. Struktur & Spaltenbeschreibung (Datenschema)

| Spaltenname | Datentyp | Schlüsseltyp | Beschreibung / Business-Logik |
| :--- | :--- | :--- | :--- |
| **`sales_order_id`** | `long` | Fremdschlüssel (FK) | Eindeutige Nummer des Kundenauftrags. |
| **`product_id`** | `string` | Fremdschlüssel (FK) | Eindeutige Identifikationsnummer des Artikels (Verknüpfung zu `dim_products`). |
| **`customer_id`** | `string` | Fremdschlüssel (FK) | Eindeutige Kundennummer (Verknüpfung zu `dim_customers`). |
| **`promotion_id`** | `string` | Fremdschlüssel (FK) | Angewendete Rabatt-ID. Standardwert bei Nichtvorhandensein: `"KEINE_PROMO"`. |
| **`order_date`** | `timestamp` | Attribut | Bestelldatum und Uhrzeit des Auftrags. |
| **`currency`** | `string` | Attribut | Währung der Transaktion (z.B. "EUR"). |
| **`order_unit`** | `string` | Attribut | Verkaufseinheit des Artikels (z.B. "Stück"). |
| **`quantity_ordered`** | `int` | Kennzahl (Metric) | Die vom Kunden gekaufte Menge des Artikels. |
| **`website_clicks`** | `int` | Kennzahl (Metric) | Anzahl der registrierten Klicks auf diesen Artikel vor dem Kauf. Standard: `0`. |
| **`gross_revenue`** | `decimal(18,2)`| Kennzahl (Metric) | **Berechnet:** `quantity_ordered * unit_price`. Der Brutto-Umsatz vor Rabatten. |
| **`discount_amount`** | `decimal(18,2)`| Kennzahl (Metric) | **Berechnet:** `gross_revenue * promotion_discount`. Der abgezogene Rabattwert in Euro. |
| **`net_revenue`** | `decimal(18,2)`| Kennzahl (Metric) | **Berechnet:** `gross_revenue - discount_amount`. Der tatsächliche, finanzrelevante Umsatz. |

---

## 3. Vorteile für das Business & BI-Reporting (Power BI / Tableau)
*   **Maximale Abfragegeschwindigkeit:** Komplexe mathematische Berechnungen (wie die prozentualen Rabattabzüge) werden einmalig beim Pipeline-Durchlauf berechnet. BI-Tools müssen Spalten wie `net_revenue` nur noch aggregieren (`SUM`), was Berichte extrem beschleunigt.
*   **Keine Rundungsfehler:** Durch die strikte Verwendung des `decimal(18,2)`-Datentyps für alle berechneten Umsatzspalten werden finanzielle Rundungsdifferenzen zwischen verschiedenen Reporting-Tools vollständig ausgeschlossen.
*   **Conversion-Analysen bereit:** Durch die direkte Gegenüberstellung von `website_clicks` und `quantity_ordered` lässt sich sofort ermitteln, wie viele Klicks im Webshop notwendig waren, um einen Kauf zu generieren.


In [26]:
from pyspark.sql.functions import col, coalesce, lit, round, sum as _sum, max as _max

# 1. Laden aller relevanten Tabellen aus dem Silver Layer
silver_items   = spark.table("Retailorg_silver.sales_order_items")
silver_orders  = spark.table("Retailorg_silver.sales_orders")
silver_promos  = spark.table("Retailorg_silver.sales_promotion")
silver_clicks  = spark.table("Retailorg_silver.sales_clicked_items")

# --- RETTUNGSSCHRITT: EINDEUTIGKEIT ERZWINGEN ---
# Wir gruppieren die Promos, damit es pro Auftrag und Produkt nur noch EINE Zeile gibt
# (Wir nehmen den maximalen Rabatt und die erste Promo-ID)
promos_unique = silver_promos.groupBy("sales_order_id", "promoted_item_id").agg(
    _max("promotion_id").alias("promotion_id"),
    _max("promotion_discount").alias("promotion_discount")
)

# Wir summieren die Klicks pro Auftrag und Produkt auf eine eindeutige Zeile
clicks_unique = silver_clicks.groupBy("sales_order_id", "product_id").agg(
    _sum("click_value").alias("click_value")
)
# ------------------------------------------------

# 2. Schritt A: Positionen mit dem Auftragskopf verbinden (Header + Items via LEFT JOIN)
sales_base = silver_items.join(
    silver_orders,
    on="sales_order_id",
    how="left"
)

# 3. Schritt B: Promotions anspielen (Jetzt über das absolut eindeutige promos_unique)
sales_with_promos = sales_base.join(
    promos_unique,
    (sales_base.sales_order_id == promos_unique.sales_order_id) & 
    (sales_base.product_id == promos_unique.promoted_item_id),
    how="left"
)

# Eindeutiges Zwischen-Select
sales_cleaned = sales_with_promos.select(
    sales_base["sales_order_id"].alias("sales_order_id"),
    sales_base["product_id"].alias("product_id"),
    sales_base["customer_id"].alias("customer_id"),
    sales_base["order_date"],
    sales_base["currency"],
    sales_base["order_unit"],
    sales_base["unit_price"],
    sales_base["quantity_ordered"],
    promos_unique["promotion_id"],
    promos_unique["promotion_discount"]
)

# 4. Schritt C: Online-Tracking-Daten anspielen (Jetzt über das eindeutige clicks_unique)
sales_enriched = sales_cleaned.join(
    clicks_unique,
    (sales_cleaned.sales_order_id == clicks_unique.sales_order_id) & 
    (sales_cleaned.product_id == clicks_unique.product_id),
    how="left"
)

# 5. Schritt D: Business-Logik & finale Gold-Berechnungen
final_gold_sales = (sales_enriched
    .select(
        sales_cleaned["sales_order_id"].alias("sales_order_id"),
        sales_cleaned["product_id"].alias("product_id"),
        sales_cleaned["customer_id"].alias("customer_id"),
        coalesce(sales_cleaned["promotion_id"], lit("KEINE_PROMO")).alias("promotion_id"),
        
        col("order_date"),
        col("currency"),
        col("order_unit"),
        col("quantity_ordered"),
        coalesce(clicks_unique["click_value"], lit(0)).alias("website_clicks"),
        
        # --- DIE GOLD-FINANZBERECHNUNGEN ---
        round(sales_cleaned["quantity_ordered"] * sales_cleaned["unit_price"], 2)
            .cast("decimal(18,2)").alias("gross_revenue"),
        
        round(sales_cleaned["quantity_ordered"] * sales_cleaned["unit_price"] * coalesce(sales_cleaned["promotion_discount"], lit(0.00)), 2)
            .cast("decimal(18,2)").alias("discount_amount"),
            
        round((sales_cleaned["quantity_ordered"] * sales_cleaned["unit_price"]) * (lit(1.00) - coalesce(sales_cleaned["promotion_discount"], lit(0.00))), 2)
            .cast("decimal(18,2)").alias("net_revenue")
    )
)

# 6. Speichern im Gold Layer
(final_gold_sales.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("Retailorg_gold.fact_sales_order_items")
)

print(f"KREUZJOIN BEHOBEN!")
print(f"Ursprüngliche Verkaufspositionen (Sollwert): {silver_items.count()}")
print(f"Finale Datensätze in Gold-Faktentabelle (Istwert): {final_gold_sales.count()}")


StatementMeta(, e210f3f9-ba26-43bf-b859-3d0953d6d8b1, 28, Finished, Available, Finished, False)

KREUZJOIN BEHOBEN!
Ursprüngliche Verkaufspositionen (Sollwert): 8137
Finale Datensätze in Gold-Faktentabelle (Istwert): 8491


In [7]:
df= spark.read.table('Retailorg_gold.fact_sales_order_items')

df=df.filter("promotion_id !='KEINE_PROMO'")
display(df.limit(5))

StatementMeta(, a45bd38e-ad16-4376-bd84-176f4cf6943f, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 19ce9dd2-a73d-47af-8b04-606b8e931566)

# Dokumentation: Gold-Faktentabelle (fact_purchase_orders)

**Layer:** Gold Layer (Business / Reporting Schicht)  
**Ziel:** Strukturierung und Bereitstellung aller B2B-Beschaffungsprozesse (Lieferantenbestellungen) in einer hochperformanten Faktentabelle. Diese Tabelle bildet die einkaufsseitige Basis des Sternschemas für Kosten- und Lieferantenanalysen.

---

## 1. Daten-Strukturierung & Prozess-Kontext
Die Tabelle übernimmt die bereinigten Bestelldaten aus dem Silver Layer und bereitet sie gezielt für betriebswirtschaftliche KPIs auf. Jede Zeile repräsentiert eine spezifische Bestellposition eines Artikels bei einem externen Lieferanten. Sie dient als direktes Gegenstück zur Verkaufsfaktentabelle (`fact_sales_order_items`), um die Beschaffungskosten den Umsätzen gegenüberzustellen.

---

## 2. Struktur & Spaltenbeschreibung (Datenschema)

| Spaltenname | Datentyp | Schlüsseltyp | Beschreibung / Business-Logik |
| :--- | :--- | :--- | :--- |
| **`purchase_order_id`** | `string` | Fremdschlüssel (FK) | Eindeutige Bestellnummer des Einkaufsbelegs. |
| **`ean13`** | `string` | Fremdschlüssel (FK) | Der 13-stellige Artikelbarcode (Hauptschlüssel zu `dim_products`). |
| **`ean5`** | `string` | Attribut | 5-stelliger Zusatzcode (z. B. für Presseartikel/Bücher). Standard: `""`. |
| **`order_timestamp`** | `timestamp` | Attribut | Exakter Zeitpunkt (Datum & Uhrzeit), an dem die Bestellung ausgelöst wurde. |
| **`order_unit`** | `string` | Attribut | Die Bestelleinheit beim Lieferanten (z. B. "Stück", "Karton", "Palette"). |
| **`purchaser_name`** | `string` | Attribut | Name des zuständigen Einkäufers oder automatischen Systems. |
| **`supplier_name`** | `string` | Attribut | Name des externen Lieferanten (Verknüpfungsschlüssel zu `dim_suppliers`). |
| **`quantity`** | `long` | Kennzahl (Metric) | Die beim Lieferanten verbindlich georderte Menge. |
| **`purchase_price_per_unit`**| `decimal(18,2)`| Kennzahl (Metric) | Der vereinbarte Netto-Einkaufspreis (EK) pro Bestelleinheit. |
| **`total_purchase_cost`** | `decimal(18,2)`| Kennzahl (Metric) | **Berechnet:** `quantity * purchase_price_per_unit`. Die gesamten finanziellen Kosten dieser Bestellposition. |

---

## 3. Vorteile für das Business & BI-Reporting (Power BI / Tableau)
*   **Absturzsicheres Berichtswesen:** Durch die im Silver Layer durchgeführte Null-Wert-Behandlung (Coalesce auf `0` bzw. `0.00`) sind alle Berechnungen innerhalb dieser Tabelle mathematisch absolut stabil.
*   **Optimale Performance:** Die Kennzahl `total_purchase_cost` ist fest materialisiert. BI-Tools müssen keine Zeilen-Multiplikationen zur Laufzeit durchführen, sondern können das Einkaufsvolumen direkt aggregieren (`SUM`).
*   **Lieferanten-Controlling:** Ermöglicht Ad-hoc-Auswertungen darüber, bei welchen Lieferanten (`supplier_name`) das höchste Einkaufsvolumen liegt und welche Einkäufer (`purchaser_name`) welche Warengruppen steuern.


In [31]:
from pyspark.sql.functions import col, round, coalesce, lit

# 1. Laden der bereinigten Einkaufsdaten aus dem Silver Layer
silver_po = spark.table("Retailorg_silver.purchase_orders")

# 2. Überführung in die Gold-Faktentabelle basierend auf Ihrem exakten Schema
gold_purchase_orders = (silver_po
    .select(
        # Schlüssel (Keys) für das Sternschema
        col("purchase_order_id"),
        col("purchase_order"),
        col("ean13"),
        coalesce(col("ean5"), lit("")).alias("ean5"),
        
        # Text-Schlüssel (Da keine IDs im Quellschema vorhanden sind)
        col("supplier_name"),   # Dient als Join-Key zu dim_suppliers
        col("purchaser_name"),  # Dient als Join-Key zu dim_employees
        
        # Attribute & Zeitstempel
        col("promotion_datetime").alias("order_timestamp"),
        col("order_unit"),
        
        # Basis-Kennzahlen (Metriken)
        col("quantity").alias("quantity_ordered"),
        col("unit_price").cast("decimal(18,2)").alias("purchase_price_per_unit"),
        
        # --- DIE GOLD-FINANZBERECHNUNG ---
        # Gesamt-Einkaufswert pro Bestellposition (Menge * Einkaufspreis)
        round(col("quantity") * col("unit_price"), 2)
            .cast("decimal(18,2)").alias("total_purchase_cost")
    )
)

# 3. Schreiben der Faktentabelle in das Gold-Schema (Delta-Format)
(gold_purchase_orders.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("Retailorg_gold.fact_purchase_orders")
)

print(f"Gold-Faktentabelle 'fact_purchase_orders' erfolgreich erstellt! Datensätze: {gold_purchase_orders.count()}")
display(gold_purchase_orders.limit(5))


StatementMeta(, e210f3f9-ba26-43bf-b859-3d0953d6d8b1, 33, Finished, Available, Finished, False)

Gold-Faktentabelle 'fact_purchase_orders' erfolgreich erstellt! Datensätze: 3973


SynapseWidget(Synapse.DataFrame, 48cf1dce-abfb-43f6-9354-0c933cc8cc5c)

In [27]:
# Test 1: Gibt es doppelte IDs im Auftragskopf?
duplicate_orders = spark.table("Retailorg_silver.sales_orders").groupBy("sales_order_id").count().filter("count > 1")
print(f"Anzahl doppelter IDs in den Auftragsköpfen: {duplicate_orders.count()}")

# Test 2: Gibt es doppelte IDs in den Positionen selbst? (Sollte 0 sein)
duplicate_items = spark.table("Retailorg_silver.sales_order_items").groupBy("sales_order_id", "product_id").count().filter("count > 1")
print(f"Anzahl doppelter Positions-Kombinationen: {duplicate_items.count()}")


StatementMeta(, e210f3f9-ba26-43bf-b859-3d0953d6d8b1, 29, Finished, Available, Finished, False)

Anzahl doppelter IDs in den Auftragsköpfen: 74
Anzahl doppelter Positions-Kombinationen: 175


# Dokumentation: Gold-Dimensionstabelle (dim_products)

**Layer:** Gold Layer (Business / Reporting Schicht)  
**Ziel:** Bereitstellung des zentralen, bereinigten Produkt- und Artikelkatalogs. Diese Tabelle dient im Sternschema als gemeinsame Dimension (*Conformed Dimension*), um Verkaufs- und Einkaufsdaten über standardisierte Produktattribute zu filtern und zu segmentieren.

---

## 1. Rolle im Sternschema (Star Schema)
Die Tabelle `dim_products` enthält alle beschreibenden Textattribute zu den Artikeln. Sie wird im Reporting-Tool (z. B. Power BI) über eine **1:N-Beziehung** mit den Faktentabellen verknüpft:
*   `dim_products.product_id` --> `fact_sales_order_items.product_id`
*   `dim_products.ean13` --> `fact_purchase_orders.ean13`

---

## 2. Struktur & Spaltenbeschreibung (Datenschema)

| Spaltenname | Datentyp | Schlüsseltyp | Beschreibung / Business-Logik |
| :--- | :--- | :--- | :--- |
| **`product_id`** | `string` | Primärschlüssel (PK) | Eindeutige, interne Artikelnummer der Retail_Org. |
| **`ean13`** | `string` | Alternativschlüssel | Internationaler 13-stelliger Barcode zur globalen Identifikation. |
| **`ean5`** | `string` | Attribut | 5-stelliger Zusatzcode (Add-On für Bücher/Presse). Standard: `""`. |
| **`product_name`** | `string` | Attribut | Klartextname des Artikels für die Anzeige im Bericht. |
| **`product_category`**| `string` | Attribut | Warengruppe / Kategorie zur Aggregation (z. B. "Lebensmittel"). |
| **`product_unit`** | `string` | Attribut | Die Basismengeneinheit des Artikels (z. B. "Stück", "Flasche"). |
| **`standard_retail_price`**| `decimal(18,2)`| Attribut | Der offizielle Standard-Listenverkaufspreis (Soll-Wert laut Regal). |

---

## 3. Vorteile für das Business-Reporting
*   **Zentrale Pflege (SCD-Bereitschaft):** Namensänderungen oder Umsortierungen von Warengruppen werden ausschließlich in dieser Tabelle durchgeführt. Die Faktentabellen bleiben davon unberührt.
*   **Soll-Ist-Vergleiche:** Durch das Mitführen des `standard_retail_price` lässt sich in Kombination mit dem tatsächlichen Umsatz (`net_revenue` aus den *Sales*) ermitteln, wie hoch die Abschriften oder Verluste durch Promotions ausgefallen sind.


In [32]:
from pyspark.sql.functions import col, trim, when

# 1. Laden der bereinigten Produktdaten aus dem Silver Layer
silver_products = spark.table("Retailorg_silver.products")

# 2. Strukturierung für das Gold Sternschema
gold_products = (silver_products
    .select(
        # Natürliche und technische Schlüssel (Keys) für die Joins
        col("product_id"),
        col("ean13"),
        col("ean5"),
        
        # Beschreibende Text-Attribute für Filter und Gruppierungen in Berichten
        col("product_name"),
        col("product_category"),
        col("product_unit"),
        
        # Der reguläre Standard-Verkaufspreis als Referenzwert (Soll-Preis)
        col("sales_price").alias("standard_retail_price")
    )
)

# 3. Schreiben der Dimensionstabelle in das Gold-Schema (Delta-Format)
(gold_products.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("Retailorg_gold.dim_products")
)

print(f"Gold-Dimensionstabelle 'dim_products' erfolgreich erstellt! Datensätze: {gold_products.count()}")
display(gold_products.limit(5))


StatementMeta(, e210f3f9-ba26-43bf-b859-3d0953d6d8b1, 34, Finished, Available, Finished, False)

Gold-Dimensionstabelle 'dim_products' erfolgreich erstellt! Datensätze: 98


SynapseWidget(Synapse.DataFrame, 3a5e2650-3404-4a70-be8a-a5e5f75e3208)

# Dokumentation: Gold-Faktentabelle (fact_active_promotions)

**Layer:** Gold Layer (Business / Reporting Schicht)  
**Ziel:** Bereitstellung eines dynamischen, tagesaktuellen Registers aller laufenden Marketing- und Rabattkampagnen. Diese Tabelle dient als operative Faktentabelle für das Live-Performance-Monitoring im Handel.

---

## 1. Funktion im Sternschema (Star Schema)
Die Tabelle `fact_active_promotions` agiert als Brücke zwischen dem aktuellen Marketing-Status und den Verkäufen. Sie ermöglicht im Berichtswesen einen direkten Filter auf Kampagnen, die am aktuellen Systemdatum (`CURRENT_DATE`) gültig sind. 
*   **Verknüpfung:** `fact_active_promotions.product_id` --> `dim_products.product_id`

---

## 2. Struktur & Spaltenbeschreibung (Datenschema)

| Spaltenname | Datentyp | Schlüsseltyp | Beschreibung / Business-Logik |
| :--- | :--- | :--- | :--- |
| **`promotion_id`** | `string` | Fremdschlüssel (FK) | Eindeutiger Identifikationscode der Werbeaktion. |
| **`product_id`** | `string` | Fremdschlüssel (FK) | ID des Artikels, für den die Promotion gilt (Match zu `dim_products`). |
| **`promotion_type`** | `string` | Attribut | Art der Kampagne (z. B. "Prozentualer Nachlass", "Mengenrabatt"). |
| **`campaign_start`** | `timestamp` | Attribut | Offizielles Startdatum der Rabattaktion. |
| **`campaign_end`** | `timestamp` | Attribut | Offizielles Enddatum der Rabattaktion. |
| **`min_units_required`** | `int` | Kennzahl | Mindestbestellmenge, damit der Rabatt greift. Standard: `1`. |
| **`total_campaign_duration_days`** | `int` | Kennzahl | Gesamtlaufzeit der Kampagne in Tagen. |
| **`absolute_discount_amount`** | `decimal(18,2)`| Kennzahl | Fester Rabattabzug in Euro (z. B. `5.00` Euro Rabatt). |
| **`percentage_discount_rate`** | `decimal(5,4)` | Kennzahl | Prozentualer Abschlag als Faktor (z. B. `0.1000` für 10% Rabatt). |

---

## 3. Operativer Nutzen für das Business
*   **Live-Filialsteuerung:** Filialleiter und E-Commerce-Manager sehen ad hoc, welche Preisanpassungen heute systemseitig aktiv geschaltet sein müssen.
*   **Betrugsprävention (Markdown Validation):** Durch den Abgleich der `min_units_required` kann direkt geprüft werden, ob Kassenrabatte fälschlicherweise auch bei geringeren Kaufmengen abgezogen wurden.


In [7]:
from pyspark.sql.functions import col, current_date, coalesce, lit

# 1. Laden der echten, bereinigten Aktionsdaten aus dem Silver Layer
# (Hinweis: Da Sie sie als 'sales_promotion' gespeichert haben, laden wir diese hier)
silver_active_promos = spark.table("Retailorg_silver.active_promotions")

# 2. Strukturierung für das Gold Layer mit den ECHTEN Spaltennamen
gold_active_promotions = (silver_active_promos
    .select(
        # Schlüssel (Keys) für das Sternschema
        col("promotion_id"),
        col("product_id"), 
        col("customer_id"),
        
        # Kampagnen-Attribute & Zeitstempel
        col("campaign_start"),
        col("campaign_end"),
        col("redemption_timestamp"),
        
        # Numerische Bedingungen und Schwellenwerte
        coalesce(col("is_eligible_flag"), lit(0)).alias("is_eligible_flag"),
        coalesce(col("min_units_required"), lit(1)).alias("min_units_required"),
        coalesce(col("discounted_quantity"), lit(0)).alias("discounted_quantity"),
        
        # Finanzielle Werte (Rabattsatz)
        col("percentage_discount_rate")
    )
)

# 3. Schreiben der Faktentabelle in das Gold-Schema (Delta-Format)
(gold_active_promotions.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("Retailorg_gold.fact_active_promotions")
)

print(f"Gold-Faktentabelle 'fact_active_promotions' erfolgreich erstellt! Aktive Kampagnen heute: {gold_active_promotions.count()}")
display(gold_active_promotions.limit(5))


StatementMeta(, c0e799b6-dd6b-45ea-a05a-569e39b7d457, 9, Finished, Available, Finished, False)

Gold-Faktentabelle 'fact_active_promotions' erfolgreich erstellt! Aktive Kampagnen heute: 423


SynapseWidget(Synapse.DataFrame, e07479c2-57bc-42e6-b29c-3068f7becd2f)

# Dokumentation: Gold-Dimensionstabelle (dim_customers)

**Layer:** Gold Layer (Business / Reporting Schicht)  
**Ziel:** Konsolidierung und Denormalisierung aller kundenbezogenen Stammdaten, Adresskomponenten und Kundenkarten-Details (*Loyalty Segments*). Diese Tabelle stellt dem Berichtswesen eine flache Struktur zur Verfügung, um Umsätze und Web-Tracking-KPIs nach Zielgruppen und Regionen zu analysieren.

---

## 1. Daten-Zusammenführung & Denormalisierung
Um die Komplexität im Reporting-Layer zu minimieren und Schneeflockenstrukturen (Snowflake Schema) zu vermeiden, führt diese Dimension zwei Datenströme aus dem Silver Layer zusammen:
*   **`customers` (Basis):** Liefert die Kerndaten des Kunden, Adressen, lückenlose Zeitstempel und Geokoordinaten.
*   **`loyalty_segments` (Erweiterung):** Liefert die kaufmännische Textbeschreibung der Kundengruppe (`loyalty_segment_description`) sowie die Mindestmenge an Käufen (`unit_threshold`), die für diesen Status notwendig war.
*   **Logik:** Der Join erfolgt als `LEFT JOIN` über das Feld `loyalty_segment` (Kunde) und `loyalty_segment_id` (Segmente). Kunden ohne Treuestatus werden automatisch mit den Platzhaltern `"KEIN_STATUS"` und `"Kein Status"` belegt.

---

## 2. Struktur & Spaltenbeschreibung (Datenschema)

| Spaltenname | Datentyp | Schlüsseltyp | Beschreibung / Business-Logik |
| :--- | :--- | :--- | :--- |
| **`customer_id`** | `string` | Primärschlüssel (PK) | Eindeutige, bereinigte Kundennummer der Retail_Org. |
| **`customer_name`** | `string` | Attribut | Vollständiger Name des Kunden (oder Platzhalter wie "Gastbesteller"). |
| **`loyalty_segment_id`** | `string` | Fremdschlüssel (FK) | Der technische Code der Statustufe (z. B. "SEG_VIP"). |
| **`loyalty_segment_name`**| `string` | Attribut | **Denormalisiert:** Die echte Textbeschreibung für Berichte (z. B. "Platin Status"). |
| **`loyalty_unit_threshold`**| `int` | Attribut | Die benötigte Anzahl an Käufen für dieses Segment. |
| **`cumulative_units_purchased`**| `int` | Kennzahl | Die historische Gesamtanzahl der vom Kunden gekauften Artikel. |
| **`postcode`** | `string` | Attribut | Bereinigte und standardisierte Postleitzahl des Wohnsitzes. |
| **`city`** | `string` | Attribut | Stadt des Kunden. |
| **`state`** | `string` | Attribut | Bundesland (wichtig für regionale Analysen). |
| **`longitude` / `latitude`** | `double` | Attribut | Exakte Geo-Koordinaten für interaktive Landkarten-Dashboards. |
| **`valid_from` / `valid_to`** | `timestamp` | Attribut | Gültigkeitszeitfenster des Datensatzes nach dem SCD-Prinzip. |

---

## 3. Vorteile für das Business & BI-Reporting
*   **Optimale Performance (No Loops):** Der Treuestatus hängt im Gold-Layer direkt als Textattribut am Kunden. Das maximiert die Performance der In-Memory-Datenbanken in Power BI, da zur Laufzeit keine Tabellen mehr gejoint werden müssen.
*   **Loyalty-Performance:** Ermöglicht die direkte Segmentierung von Finanzdaten: *„Welche Kundensegmente (`loyalty_segment_name`) generieren den höchsten Netto-Umsatz (`net_revenue`) im Onlineshop?“*


In [11]:
from pyspark.sql.functions import col, coalesce, lit

# 1. Laden der beiden bereinigten Silver-Tabellen
silver_customers = spark.table("Retailorg_silver.customers")
silver_segments  = spark.table("Retailorg_silver.loyalty_segments")

# 2. Zusammenfügen (Join) über die Segment-ID
# Wir nutzen einen Left-Join, damit Kunden ohne Treueprogramm (Gastbesteller) nicht gelöscht werden
customers_joined = silver_customers.join(
    silver_segments,
    silver_customers.loyalty_segment == silver_segments.loyalty_segment_id,
    how="left"
)

# 3. Strukturierung für das finale Gold-Sternschema (Denormalisierung)
gold_customers = (customers_joined
    .select(
        # Primärschlüssel (Primary Key) für das Sternschema
        silver_customers["customer_id"].alias("customer_id"),
        silver_customers["customer_name"],
        
        # --- DIE ZUSAMMENGEFÜHRTEN MARKETING-DATEN ---
        # Wir nehmen die ID und die Klartextbeschreibung aus der Segmente-Tabelle
        coalesce(silver_customers["loyalty_segment"], lit("KEIN_STATUS")).alias("loyalty_segment_id"),
        coalesce(silver_segments["loyalty_segment_description"], lit("Kein Status")).alias("loyalty_segment_name"),
        coalesce(silver_segments["unit_threshold"], lit(0)).alias("loyalty_unit_threshold"),
        
        # Historische Kaufmetrik des Kunden
        coalesce(silver_customers["units_purchased"], lit(0)).alias("cumulative_units_purchased"),
        
        # Adress- und Regionaldaten für Geo-Filter im Dashboard
        silver_customers["street"],
        silver_customers["house_number"],
        silver_customers["unit"].alias("building_unit"),
        silver_customers["postcode"],
        silver_customers["city"],
        silver_customers["state"],
        silver_customers["region"],
        silver_customers["district"],
        silver_customers["ship_to_address"],
        
        # Geo-Koordinaten für interaktive Karten-Visualisierungen
        silver_customers["longitude"],
        silver_customers["latitude"],
        
        # Steuerdaten
        silver_customers["tax_id"],
        silver_customers["tax_code"],
        
        # Gültigkeitszeiträume für historische Abgleiche (SCD) aus der Kundentabelle
        silver_customers["valid_from"],
        silver_customers["valid_to"]
    )
)

# 4. Schreiben der Dimensionstabelle in das Gold-Schema (Delta-Format)
(gold_customers.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("Retailorg_gold.dim_customers")
)

print(f"Gold-Dimensionstabelle 'dim_customers' (inkl. Loyalty-Join) erfolgreich erstellt! Datensätze: {gold_customers.count()}")
display(gold_customers.limit(5))


StatementMeta(, c0e799b6-dd6b-45ea-a05a-569e39b7d457, 13, Finished, Available, Finished, False)

Gold-Dimensionstabelle 'dim_customers' (inkl. Loyalty-Join) erfolgreich erstellt! Datensätze: 28813


SynapseWidget(Synapse.DataFrame, f3fcf2aa-5f8d-45fe-91cc-c0a631b23845)

# Dokumentation: Gold-Dimensionstabelle (dim_suppliers)

**Layer:** Gold Layer (Business / Reporting Schicht)  
**Ziel:** Bereitstellung des zentralen Lieferantenstamms inklusive aller Adressdaten, geografischen Koordinaten und Warengruppen-Klassifizierungen. Diese Tabelle dient im Sternschema dazu, die B2B-Beschaffungsprozesse und Einkaufskosten nach Lieferantenmerkmalen zu segmentieren.

---

## 1. Rolle im Sternschema (Star Schema)
Die Tabelle `dim_suppliers` enthält ausschließlich beschreibende Textattribute und Geo-Daten zu den Handelspartnern. Sie enthält **keine beweglichen Kennzahlen** und wird im Reporting-Tool (z. B. Power BI) über eine **1:N-Beziehung** mit der einkaufsseitigen Faktentabelle verknüpft:
*   `dim_suppliers.supplier_name` --> `fact_purchase_orders.supplier_name`

---

## 2. Struktur & Spaltenbeschreibung (Datenschema)

| Spaltenname | Datentyp | Schlüsseltyp | Beschreibung / Business-Logik |
| :--- | :--- | :--- | :--- |
| **`supplier_id`** | `string` | Primärschlüssel (PK) | Eindeutige Kreditoren- bzw. Lieferantennummer. |
| **`supplier_name`** | `string` | Fremdschlüssel (FK) | Name des Lieferanten (Verknüpfungsschlüssel zu `fact_purchase_orders`). |
| **`tax_id`** | `string` | Attribut | Offizielle Steuernummer des Partners für Compliance-Zwecke. |
| **`supplier_category`**| `string` | Attribut | **Denormalisiert:** Die vom Lieferanten bereitgestellten Warengruppen (z. B. "Getränke"). |
| **`postcode`** | `string` | Attribut | Bereinigte Postleitzahl des Hauptlagers/Firmensitzes. |
| **`city`** | `string` | Attribut | Stadt, in der der Lieferant ansässig ist. |
| **`state`** | `string` | Attribut | Bundesland (wichtig für regionale Beschaffungsanalysen). |
| **`longitude` / `latitude`** | `double` | Attribut | Präzise GPS-Koordinaten für interaktive Landkarten-Dashboards. |

---

## 3. Analytischer Nutzen für das Business
*   **Geografisches Logistik-Controlling:** Durch die Mitführung von `longitude` und `latitude` kann das BI-Tool das gesamte Einkaufsvolumen (`total_purchase_cost` aus den *Purchase Orders*) auf einer Landkarte visualisieren. Das deckt sofort auf, aus welchen Regionen die Retail_Org den Großteil ihrer Waren bezieht.
*   **Cluster nach Warengruppen:** Einkaufsleiter können Berichte direkt nach `supplier_category` filtern, um das Einkaufsvolumen und die Lieferantenabhängigkeit pro Sortimentsbereich zu überwachen.


In [12]:
from pyspark.sql.functions import col, coalesce, lit

# 1. Laden der bereinigten Lieferantendaten aus dem Silver Layer
silver_suppliers = spark.table("Retailorg_silver.suppliers")

# 2. Strukturierung für das Gold Sternschema
gold_suppliers = (silver_suppliers
    .select(
        # Primärschlüssel (Primary Key) für das Sternschema
        col("supplier_id"),
        
        # Text-Schlüssel für den Join mit der Einkaufs-Faktentabelle
        col("supplier_name"),
        
        # Steuerdaten
        col("tax_id"),
        
        # Adress- und Regionaldaten für geografische Filterungen in Dashboards
        col("street"),
        col("house_number"),
        col("unit").alias("building_unit"),
        col("postcode"),
        col("city"),
        col("state"),
        col("region"),
        col("district"),
        
        # Fachliches Zuordnungsmerkmal (z. B. Warengruppen)
        col("items_provided").alias("supplier_category"),
        
        # Geo-Koordinaten für interaktive Landkarten-Visualisierungen
        col("longitude"),
        col("latitude")
    )
)

# 3. Schreiben der Dimensionstabelle in das Gold-Schema (Delta-Format)
(gold_suppliers.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("Retailorg_gold.dim_suppliers")
)

print(f"Gold-Dimensionstabelle 'dim_suppliers' erfolgreich erstellt! Datensätze: {gold_suppliers.count()}")
display(gold_suppliers.limit(5))


StatementMeta(, c0e799b6-dd6b-45ea-a05a-569e39b7d457, 14, Finished, Available, Finished, False)

Gold-Dimensionstabelle 'dim_suppliers' erfolgreich erstellt! Datensätze: 15


SynapseWidget(Synapse.DataFrame, 60834290-ca7b-4edc-be46-12fae3d96406)

# Dokumentation: Gold-Dimensionstabelle (dim_employees)

**Layer:** Gold Layer (Business / Reporting Schicht)  
**Ziel:** Bereitstellung des zentralen, historisierten Mitarbeiterstamms. Diese Tabelle dient im Sternschema als Dimension, um operative Kennzahlen der Retail_Org nach Abteilungen, Mitarbeiterregionen und Zuständigkeiten aufzuschneiden.

---

## 1. Rolle im Sternschema (Star Schema)
Die Tabelle `dim_employees` enthält ausschließlich beschreibende Textattribute und Historisierungsmerkmale der Mitarbeiter. Sie wird im Reporting-Tool (z. B. Power BI) über eine **1:N-Beziehung** mit den Transaktionsdaten verknüpft:
*   `dim_employees.employee_name` --> `fact_purchase_orders.purchaser_name`

---

## 2. Struktur & Spaltenbeschreibung (Datenschema)

| Spaltenname | Datentyp | Schlüsseltyp | Beschreibung / Business-Logik |
| :--- | :--- | :--- | :--- |
| **`employee_id`** | `string` | Primärschlüssel (PK) | Eindeutige, systemübergreifende Personalnummer. |
| **`employee_key`** | `string` | Alternativschlüssel | Technischer Schlüssel des Mitarbeiters aus dem Quellsystem. |
| **`employee_name`** | `string` | Fremdschlüssel (FK) | Name des Mitarbeiters (Verknüpfungsschlüssel zu `fact_purchase_orders`). |
| **`department`** | `string` | Attribut | Abteilung, in der der Mitarbeiter tätig ist (z. B. "Einkauf", "Filiale 01"). |
| **`employee_region`** | `string` | Attribut | Regionale Zuordnung des Arbeitsplatzes. |
| **`is_active_employee`**| `boolean` | Attribut | Status-Flag: Gibt an, ob der Mitarbeiter aktuell aktiv im Unternehmen ist (`True`/`False`). |
| **`valid_from` / `valid_to`**| `timestamp` | Attribut | Gültigkeitszeitraum des Datensatzes nach dem SCD-Prinzip (Slowly Changing Dimensions). |

---

## 3. Analytischer Nutzen für das Business
*   **Performance nach Abteilungen:** Ermöglicht die direkte Auswertung von Einkaufsvolumen (`total_purchase_cost`) auf die verschiedenen Abteilungen (`department`), um Budgets zu überwachen.
*   **SCD-Historisierung:** Durch die Mitführung von `valid_from` und `valid_to` wird sichergestellt, dass bei einem Abteilungswechsel eines Einkäufers die historischen Bestellungen korrekt der *alten* Abteilung zugeordnet bleiben, während neue Bestellungen in der *neuen* Abteilung einlaufen.


In [13]:
from pyspark.sql.functions import col, coalesce, lit

# 1. Laden der bereinigten Mitarbeiterdaten aus dem Silver Layer
silver_employees = spark.table("Retailorg_silver.company_employees")

# 2. Strukturierung für das Gold Sternschema
gold_employees = (silver_employees
    .select(
        # Primärschlüssel für das Sternschema
        col("employee_id"),
        col("employee_key"),
        
        # Text-Schlüssel für den Join mit der Faktentabelle (Purchaser-Name)
        col("employee_name"),
        
        # Organisations- und Filtermerkmale
        col("department"),
        col("region").alias("employee_region"),
        
        # Gültigkeits- und Statusflags (SCD-Eigenschaften)
        col("active_record").alias("is_active_employee"),
        col("active_record_start").alias("valid_from"),
        col("active_record_end").alias("valid_to")
    )
)

# 3. Schreiben der Dimensionstabelle in das Gold-Schema (Delta-Format)
(gold_employees.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("Retailorg_gold.dim_employees")
)

print(f"Gold-Dimensionstabelle 'dim_employees' erfolgreich erstellt! Datensätze: {gold_employees.count()}")
display(gold_employees.limit(5))


StatementMeta(, c0e799b6-dd6b-45ea-a05a-569e39b7d457, 15, Finished, Available, Finished, False)

Gold-Dimensionstabelle 'dim_employees' erfolgreich erstellt! Datensätze: 500


SynapseWidget(Synapse.DataFrame, ce457117-b2dc-420e-a56a-b60822c48700)

# Dokumentation: Gold-Dimensionstabelle (dim_promotions)

**Layer:** Gold Layer (Business / Reporting Schicht)  
**Ziel:** Bereitstellung des zentralen Katalogs aller jemals angelegten Marketing- und Rabattkampagnen. Diese Tabelle dient im Sternschema als Dimension, um absolute Verkaufszahlen, gewährte Rabatte und Conversion-Rates nach Kampagnenarten und Sortimentsbereichen zu segmentieren.

---

## 1. Rolle im Sternschema (Star Schema)
Die Tabelle `dim_promotions` enthält ausschließlich beschreibende Textattribute und geplante Laufzeiten zu den Marketingaktivitäten. Sie ist über eine **1:N-Beziehung** direkt mit den Faktentabellen verknüpft:
*   `dim_promotions.promotion_id` --> `fact_sales_order_items.promotion_id`
*   `dim_promotions.promotion_id` --> `fact_active_promotions.promotion_id`

---

## 2. Struktur & Spaltenbeschreibung (Datenschema)

| Spaltenname | Datentyp | Schlüsseltyp | Beschreibung / Business-Logik |
| :--- | :--- | :--- | :--- |
| **`promotion_id`** | `string` | Primärschlüssel (PK) | Eindeutiger Identifikationscode der Werbeaktion. |
| **`promotion_type`** | `string` | Attribut | Art der Kampagne (z. B. "Prozentualer Nachlass", "Mengenrabatt"). |
| **`qualifying_product_group`**| `string` | Attribut | Angabe, für welche Sortimentsteile die Aktion gilt (z. B. "Alle", "Kaffee"). |
| **`free_product_ids`** | `string` | Attribut | Auflistung von Gratisbeigaben (z. B. bei "Nimm 3, zahl 2"). Standard: `"Keine"`. |
| **`units_required_threshold`**| `int` | Attribut | Mindestbestellmenge im Warenkorb, damit die Aktion greift. |
| **`planned_duration_days`** | `int` | Attribut | Die vom Marketing geplante Laufzeit der Aktion in Tagen. |
| **`standard_dollar_discount`**| `decimal(18,2)`| Attribut | Der vordefinierte absolute Rabattwert laut Stammdatenblatt. |
| **`standard_percent_discount`**| `decimal(5,4)` | Attribut | Der vordefinierte prozentuale Rabattsatz laut Stammdatenblatt. |
| **`campaign_start`** | `timestamp` | Attribut | Offizieller Startzeitpunkt der Kampagnengültigkeit. |
| **`campaign_end`** | `timestamp` | Attribut | Offizieller Endzeitpunkt (Deadline) der Kampagnengültigkeit. |

---

## 3. Analytischer Nutzen für das Business
*   **Soll-Ist-Abgleich der Rabatte:** Da hier die vordefinierten Rabattsätze (`standard_percent_discount`) liegen, kann das Controlling im Dashboard prüfen, ob die an den Kassen tatsächlich abgezogenen Rabatt-Beträge exakt den Vorgaben des Marketings entsprachen.
*   **ROI nach Kampagnenart:** Ermöglicht die Gruppierung des Netto-Umsatzes nach `promotion_type`, um zu ermitteln, welche Rabattform die höchste Kundenresonanz erzielt.


In [16]:
from pyspark.sql.functions import col, coalesce, lit

# 1. Laden der echten Promotions-Stammdaten aus dem Silver Layer
silver_promotions_master = spark.table("Retailorg_silver.promotions")

# 2. Strukturierung für das Gold Sternschema basierend auf Ihrem exakten Schema
gold_promotions = (silver_promotions_master
    .select(
        # Primärschlüssel (Primary Key) für das Sternschema
        col("promotion_id"),
        
        # Beschreibende Text-Attribute und Produktzuordnungen
        col("promotion_type"),
        col("qualifying_products").alias("qualifying_product_group"),
        col("free_product_ids"),
        
        # Bedingungen und Laufzeiten (Stammdaten-Attribute)
        coalesce(col("units_required"), lit(1)).alias("units_required_threshold"),
        col("campaign_length_days").alias("planned_duration_days"),
        
        # Vordefinierte Soll-Rabatte für Soll-Ist-Vergleiche
        coalesce(col("dollar_discount"), lit(0.00)).cast("decimal(18,2)").alias("standard_dollar_discount"),
        coalesce(col("percent_discount"), lit(0.0000)).cast("decimal(5,4)").alias("standard_percent_discount"),
        
        # Gültigkeitszeiträume der Kampagne
        col("valid_from").alias("campaign_start"),
        col("valid_to").alias("campaign_end")
    )
    # Sicherstellen, dass jede Kampagnen-ID nur genau EINMAL im Katalog existiert
    .dropDuplicates(["promotion_id"])
)

# 3. Schreiben der Dimensionstabelle in das Gold-Schema (Delta-Format)
(gold_promotions.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("Retailorg_gold.dim_promotions")
)

print(f"Gold-Dimensionstabelle 'dim_promotions' erfolgreich erstellt! Eindeutige Kampagnen: {gold_promotions.count()}")
display(gold_promotions.limit(5))


StatementMeta(, c0e799b6-dd6b-45ea-a05a-569e39b7d457, 18, Finished, Available, Finished, False)

Gold-Dimensionstabelle 'dim_promotions' erfolgreich erstellt! Eindeutige Kampagnen: 4


SynapseWidget(Synapse.DataFrame, 88f729d7-82f2-4f4f-8eaf-60fbdd815240)